# Week 3 — Cleaning your data
## group4b · Ride-hailing trips

**Your question**
> Which pickup zones generate the most revenue, and where are cancellations concentrated?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [ ]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)


pandas 2.2.3


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [ ]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group4b"
SCHEMA_RAW   = "raw_rides"
SCHEMA_MINE  = "group4b"

password = getpass("Password for group4b: ")

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.trips", engine)


Password for group4b: ··········


,rows
0,28112


You should see **28,112**. If not, stop — something is

wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [ ]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.trips", engine)
drivers = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.drivers", engine)
riders = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.riders", engine)
zones = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.zones", engine)

print('trips   ', df.shape)
print('drivers'.ljust(12), drivers.shape)
print('riders'.ljust(12), riders.shape)
print('zones'.ljust(12), zones.shape)


trips    (28112, 10)
drivers      (200, 5)
riders       (1000, 4)
zones        (15, 3)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [ ]:
df.head(10)


,trip_id,trip_date,driver_id,rider_id,pickup_zone_id,dropoff_zone_id,distance_km,duration_min,status,fare
0,6417,2024-06-01,126,78,6,12,7.91,33.5,Completed,21.35
1,6365,14-Dec-2025,46,429,12,2,7.98,32.2,Completed,28.84
2,1615,2024-12-14,171,641,11,14,2.96,12.3,Cancelled by rider,13.13
3,26588,2025/12/06,141,58,2,10,21.51,62.2,Cancelled by rider,59.86
4,3289,2024-03-02,194,80,12,13,2.72,16.3,Completed,12.13
5,2534,2024-09-21,28,413,5,15,3.11,12.4,Completed,15.59
6,8449,2025-01-08,119,829,2,14,8.32,27.0,Completed,26.71
7,3340,14-Nov-2024,69,376,7,10,4.42,18.5,Completed,15.01
8,8911,2025-10-23,122,969,2,15,2.97,13.5,Completed,12.69
9,6895,2024-10-31,21,461,4,4,9.00,32.8,Completed,33.52


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28112 entries, 0 to 28111
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   trip_id          28112 non-null  int64  
 1   trip_date        28112 non-null  object 
 2   driver_id        28112 non-null  int64  
 3   rider_id         28112 non-null  int64  
 4   pickup_zone_id   28112 non-null  int64  
 5   dropoff_zone_id  28112 non-null  int64  
 6   distance_km      28112 non-null  float64
 7   duration_min     28112 non-null  float64
 8   status           28112 non-null  object 
 9   fare             26706 non-null  float64
dtypes: float64(3), int64(5), object(2)
memory usage: 2.1+ MB


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [ ]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['trip_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['status'].nunique(),
}
before


{'rows': 28112, 'duplicates': 112, 'missing': np.int64(1406), 'categories': 12}

---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [ ]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))


before: 28112
after:  28000
removed: 112


**TODO — write down the number removed. Does it match your
week 2 figure?**

If it does not, you are looking at something different from what you counted.
Work out which before continuing.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `status`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [ ]:
# what it looks like now
df['status'].value_counts(dropna=False)


,count
status,
Completed,21072
Cancelled by rider,2211
Cancelled by driver,1250
COMPLETED,1037
Completed,966
completed,958
Cancelled by rider,124
CANCELLED BY RIDER,111
cancelled by rider,101


In [ ]:
df['status'] = df['status'].str.strip().str.title()

df['status'].value_counts(dropna=False)


,count
status,
Completed,24033
Cancelled By Rider,2547
Cancelled By Driver,1420


**TODO — how many categories now, and how many before?**

Now do the same for the other text columns. Week 2 should have told you which
ones are affected — it is not only this one.


In [ ]:
# TODO: check and clean the text columns in your dimension tables
drivers['vehicle_type'] = drivers['vehicle_type'].str.strip().str.title()
riders['payment_preference'] = riders['payment_preference'].str.strip().str.title()

# check your work
drivers['vehicle_type'].value_counts(dropna=False).head(15)


,count
vehicle_type,
Saloon,113
Shared,47
Suv,40


---
## 6. Parse the dates

`trip_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.


In [ ]:
# what formats are present
df['trip_date'].str.len().value_counts()


,count
trip_date,
10,26302
11,1698


In [ ]:
df['trip_date'] = pd.to_datetime(
    df['trip_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'      # anything unparseable becomes NaT rather than crashing
)

print("could not parse:", df['trip_date'].isna().sum())
print("range:", df['trip_date'].min(), "to", df['trip_date'].max())


could not parse: 0
range: 2024-01-01 00:00:00 to 2025-12-30 00:00:00


**TODO — does that date range make sense now?**

Compare it with what the text version gave you in week 2. This is where the
text-sorting problem finally goes away.

If any rows failed to parse, decide what to do with them and say why.


---
## 7. Deal with impossible values

Week 2 found zero fares on completed trips. Look at them before deciding.


In [ ]:
bad = df[(df['fare'] == 0) & (df['status'].str.strip().str.upper() == 'COMPLETED')]
print("rows affected:", len(bad))
bad.head(10)


rows affected: 38


,trip_id,trip_date,driver_id,rider_id,pickup_zone_id,dropoff_zone_id,distance_km,duration_min,status,fare
220,12648,2024-06-26,18,13,3,8,5.16,19.3,Completed,0.0
2249,17589,2025-05-03,49,539,15,7,25.76,100.2,Completed,0.0
2524,15144,2025-12-16,181,401,3,4,7.63,26.9,Completed,0.0
3578,14241,2024-09-21,60,748,12,12,11.61,43.5,Completed,0.0
7816,20760,2024-06-17,155,933,2,13,3.05,17.2,Completed,0.0
7849,18682,2025-03-17,41,50,13,11,17.88,49.4,Completed,0.0
8125,5529,2025-01-18,185,834,8,15,5.17,15.2,Completed,0.0
8502,7397,2024-02-19,197,881,6,9,1.46,10.7,Completed,0.0
8997,6877,2025-02-06,194,984,15,7,5.79,23.3,Completed,0.0
9169,5026,2024-10-16,109,409,3,13,3.11,9.8,Completed,0.0


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong
one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the
   absolute value may be justified. Only if you can argue it.


In [ ]:
# TODO: implement your decision. One of these, or your own.

# option 1 — drop
# df = df[~df.index.isin(bad.index)]

# option 2 — set to NULL
# df.loc[bad.index, 'COLUMN'] = np.nan

print("rows now:", len(df))


rows now: 28000


---
## 8. Deal with orphan keys

Some `driver_id` values in your fact table point at
`drivers` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [ ]:
valid = set(drivers['driver_id'])
orphans = df[~df['driver_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['trip_id', 'driver_id']].head(10)


orphan rows: 70


,trip_id,driver_id
183,12318,90011
1167,12266,90046
1525,26509,90056
2242,7212,90025
2790,10336,90013
2909,3766,90012
3251,11202,90026
3796,24973,90004
4109,9851,90066
4812,2220,90027


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and
   your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.


In [ ]:
# TODO: implement your decision

# option 1 — drop
# df = df[df['driver_id'].isin(valid)]

# option 2 — add an Unknown row to the dimension, then repoint orphans at it
# unknown = pd.DataFrame([{'driver_id': -1}])
# drivers = pd.concat([drivers, unknown], ignore_index=True)
# df.loc[~df['driver_id'].isin(valid), 'driver_id'] = -1

print("rows now:", len(df))


rows now: 28000


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [ ]:
df.isna().sum().sort_values(ascending=False)


,0
fare,1400
trip_id,0
trip_date,0
driver_id,0
pickup_zone_id,0
rider_id,0
dropoff_zone_id,0
distance_km,0
duration_min,0
status,0


**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `fare` | | | |
| | | | |

Options: leave as NULL (honest, Power BI shows blanks), fill with a label like
`'Unknown'` (good for text you will group by), or drop the row (only if the row
is useless without it).


In [ ]:
# TODO: implement your decisions

# example — label missing text so it groups properly in Power BI
# df['fare'] = df['fare'].fillna('Unknown')

df.isna().sum().sort_values(ascending=False).head()


,0
fare,1400
trip_id,0
trip_date,0
driver_id,0
pickup_zone_id,0


---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [ ]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['trip_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['status'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


,rows,duplicates,missing,categories
before,28112,112,1406,12
after,28000,0,1400,3


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates,
how many were impossible values, and how many were orphans. If the numbers do
not add up, something happened that you did not intend.


---
## 11. Write the clean tables back

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds the
table each time you run the notebook — use `'append'` by mistake and running
twice silently doubles your data.


In [ ]:
df.to_sql('trips_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
drivers.to_sql('drivers_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
riders.to_sql('riders_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
zones.to_sql('zones_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)

print("written")


written


### Confirm they landed


In [ ]:
pd.read_sql(f"""
    SELECT table_name,
           (xpath('/row/c/text()',
            query_to_xml(format('SELECT count(*) AS c FROM %%I.%%I',
            table_schema, table_name), false, true, '')))[1]::text::int AS rows
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA_MINE}'
    ORDER BY table_name
""", engine)


,table_name,rows
0,drivers_clean,200
1,riders_clean,1000
2,trips_clean,28000
3,zones_clean,15


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group4b` and build the model on these tables.
